# MLflow RAGAS Scores Query

This notebook fetches `ragas_scores.json` artifacts from MLflow runs and computes mean scores grouped by:
- `question_class`
- `subdomain`
- (`question_class`, `subdomain`)

In [1]:
import json
import os
from pathlib import Path

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

/home/mahee/Work/Thesis/Repos/langchain-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Config
MLFLOW_TRACKING_URI = "http://localhost:8567"
EXPERIMENT_NAME =  "langchain-rag-evl" #None  # Set to a name string to filter one experiment
MAX_RUNS = 200

# Use one of: "latest", "all", or a specific run_id
RUN_SELECTION = "latest"

RAGAS_TABLE_ARTIFACT = "ragas_scores.json"
METRIC_COLS = [
    "faithfulness",
    "context_precision",
    "context_recall",
    "answer_relevance",
    "factual_correctness",
]

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Tracking URI: http://localhost:8567


In [3]:
def list_artifacts_recursive(run_id: str, path: str = ""):
    items = []
    for art in client.list_artifacts(run_id, path):
        items.append(art.path)
        if art.is_dir:
            items.extend(list_artifacts_recursive(run_id, art.path))
    return items


def normalize_logged_table(json_obj) -> pd.DataFrame:
    # mlflow.log_table can be stored in different JSON shapes depending on version.
    if isinstance(json_obj, list):
        return pd.DataFrame(json_obj)

    if isinstance(json_obj, dict):
        if "data" in json_obj and "columns" in json_obj:
            return pd.DataFrame(json_obj["data"], columns=json_obj["columns"])
        return pd.DataFrame(json_obj)

    raise ValueError("Unsupported ragas_scores.json structure")


def load_ragas_table_for_run(run_id: str):
    artifact_paths = list_artifacts_recursive(run_id)

    # Find ragas_scores.json at any artifact depth.
    matches = [p for p in artifact_paths if p.endswith(RAGAS_TABLE_ARTIFACT)]
    if not matches:
        return None, None

    artifact_path = matches[0]
    local_path = client.download_artifacts(run_id, artifact_path)

    with open(local_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    df = normalize_logged_table(payload)
    return df, artifact_path

In [4]:
# Discover runs
if EXPERIMENT_NAME:
    exp = client.get_experiment_by_name(EXPERIMENT_NAME)
    if exp is None:
        raise ValueError(f"Experiment not found: {EXPERIMENT_NAME}")
    experiment_ids = [exp.experiment_id]
else:
    experiment_ids = [e.experiment_id for e in client.search_experiments()]

runs_df = mlflow.search_runs(
    experiment_ids=experiment_ids,
    order_by=["attribute.start_time DESC"],
    max_results=MAX_RUNS,
)

if runs_df.empty:
    raise ValueError("No MLflow runs found for the current filter.")

print(f"Found {len(runs_df)} runs")
runs_df[["run_id", "experiment_id", "start_time", "status"]].head(10)

Found 19 runs


,run_id,experiment_id,start_time,status
0,026c927478684bb88226e8b64d58c729,17,2026-04-13 13:04:22.096000+00:00,FINISHED
1,e266d6a95f154ac4865df26859ca7d6c,17,2026-04-13 12:14:37.418000+00:00,FINISHED
2,9304cbdc75f4454c80d9401f8c49ad5f,17,2026-04-13 10:55:58.932000+00:00,FINISHED
3,ed3bbb2eded640d7a646a68a7767731f,17,2026-04-13 10:51:12.594000+00:00,FINISHED
4,dbe4d2710ab6425ca49d04dd56da92ba,17,2026-04-13 10:36:00.346000+00:00,FINISHED
5,13a96e77742a471eb9f8127ac9e0bca3,17,2026-04-13 08:37:15.417000+00:00,FINISHED
6,b3d733c2cf514777afb0eb74c8278fd9,17,2026-04-13 07:31:34.077000+00:00,FINISHED
7,28e13a4242374df5af4cea82784f55f5,17,2026-04-11 12:27:12.220000+00:00,FINISHED
8,fbda50f2699847258257e50a6bd9fcfa,17,2026-04-11 12:14:33.824000+00:00,FINISHED
9,e237dc68bf4b49a5a31c86d3d7baa73e,17,2026-04-08 16:16:25.539000+00:00,FINISHED


In [5]:
# Pull ragas_scores.json from each run
tables = []
misses = []

for run_id in runs_df["run_id"].tolist():
    df, artifact_path = load_ragas_table_for_run(run_id)
    if df is None:
        misses.append(run_id)
        continue

    df = df.copy()
    df["run_id"] = run_id
    df["artifact_path"] = artifact_path
    tables.append(df)

if not tables:
    raise ValueError("No ragas_scores.json artifacts found in discovered runs.")

all_scores = pd.concat(tables, ignore_index=True)

for col in METRIC_COLS:
    if col in all_scores.columns:
        all_scores[col] = pd.to_numeric(all_scores[col], errors="coerce")

print(f"Runs with ragas_scores.json: {all_scores['run_id'].nunique()}")
print(f"Runs without ragas_scores.json: {len(misses)}")
all_scores.head()

Runs with ragas_scores.json: 19
Runs without ragas_scores.json: 0


,user_input,response,reference,subdomain,question_class,faithfulness,context_precision,context_recall,answer_relevance,factual_correctness,metric_errors,run_id,artifact_path
0,What does the acronym NISQ stand for and what ...,NISQ stands for Noisy Intermediate-Scale Quant...,NISQ stands for Noisy Intermediate-Scale Quant...,nisq_constraints_and_qsd_constraints,fact_single,1.0,0.679167,1.000000,0.772702,0.57,{},026c927478684bb88226e8b64d58c729,ragas_scores.json
1,How does the focus of data collection shift as...,The focus of data collection shifts from prima...,"In the early stages of development, quantum pr...",nisq_constraints_and_qsd_constraints,summary,1.0,0.804167,1.000000,0.972523,0.67,{},026c927478684bb88226e8b64d58c729,ragas_scores.json
2,Based on the constraints of near-term quantum ...,The noise in quantum gates will limit the size...,There is a strict ceiling on circuit size beca...,nisq_constraints_and_qsd_constraints,reasoning,1.0,0.679167,0.666667,0.753315,0.62,{},026c927478684bb88226e8b64d58c729,ragas_scores.json
3,What is the most popular framework for develop...,I don't know. The context provided does not me...,The provided documents do not contain sufficie...,nisq_constraints_and_qsd_constraints,unanswerable,0.8,0.000000,1.000000,0.000000,0.67,{},026c927478684bb88226e8b64d58c729,ragas_scores.json
4,What are the three types of data that can be l...,The three types of data that can be logged per...,"Parameters (key-value pairs), metrics (quantit...",experiment_tracking_fundamentals,fact_single,0.0,0.000000,0.000000,1.000000,0.40,{},026c927478684bb88226e8b64d58c729,ragas_scores.json


In [6]:
# Select target rows for aggregation
if RUN_SELECTION == "latest":
    latest_run_id = runs_df.iloc[0]["run_id"]
    target = all_scores[all_scores["run_id"] == latest_run_id].copy()
    print(f"Using latest run: {latest_run_id}")
elif RUN_SELECTION == "all":
    target = all_scores.copy()
    print("Using all runs with ragas_scores.json")
else:
    target = all_scores[all_scores["run_id"] == RUN_SELECTION].copy()
    if target.empty:
        raise ValueError(f"No rows found for run_id={RUN_SELECTION}")
    print(f"Using selected run: {RUN_SELECTION}")

required_cols = ["question_class", "subdomain"]
for c in required_cols:
    if c not in target.columns:
        raise ValueError(f"Missing required column in ragas table: {c}")

target[["run_id", "user_input", "question_class", "subdomain"] + [c for c in METRIC_COLS if c in target.columns]].head()

Using latest run: 026c927478684bb88226e8b64d58c729


,run_id,user_input,question_class,subdomain,faithfulness,context_precision,context_recall,answer_relevance,factual_correctness
0,026c927478684bb88226e8b64d58c729,What does the acronym NISQ stand for and what ...,fact_single,nisq_constraints_and_qsd_constraints,1.0,0.679167,1.000000,0.772702,0.57
1,026c927478684bb88226e8b64d58c729,How does the focus of data collection shift as...,summary,nisq_constraints_and_qsd_constraints,1.0,0.804167,1.000000,0.972523,0.67
2,026c927478684bb88226e8b64d58c729,Based on the constraints of near-term quantum ...,reasoning,nisq_constraints_and_qsd_constraints,1.0,0.679167,0.666667,0.753315,0.62
3,026c927478684bb88226e8b64d58c729,What is the most popular framework for develop...,unanswerable,nisq_constraints_and_qsd_constraints,0.8,0.000000,1.000000,0.000000,0.67
4,026c927478684bb88226e8b64d58c729,What are the three types of data that can be l...,fact_single,experiment_tracking_fundamentals,0.0,0.000000,0.000000,1.000000,0.40


In [7]:
metric_cols_present = [c for c in METRIC_COLS if c in target.columns]

avg_by_question_class = (
    target.groupby("question_class", dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

avg_by_subdomain = (
    target.groupby("subdomain", dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

avg_by_qclass_and_subdomain = (
    target.groupby(["question_class", "subdomain"], dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

print("Average by question_class")
display(avg_by_question_class)

print("Average by subdomain")
display(avg_by_subdomain)

print("Average by (question_class, subdomain)")
display(avg_by_qclass_and_subdomain)

Average by question_class


,faithfulness,context_precision,context_recall,answer_relevance,factual_correctness
question_class,,,,,
fact_single,0.600000,0.202500,0.400000,0.475463,0.252
reasoning,0.525000,0.486944,0.733333,0.804132,0.612
summary,0.951282,0.627500,0.750000,0.829863,0.334
unanswerable,0.694615,0.000000,0.600000,0.106021,0.400


Average by subdomain


,faithfulness,context_precision,context_recall,answer_relevance,factual_correctness
subdomain,,,,,
experiment_tracking_fundamentals,0.439103,0.250000,0.437500,0.756063,0.2000
mlflow_tracking_api,0.718750,0.451389,0.666667,0.337497,0.4375
nisq_constraints_and_qsd_constraints,0.950000,0.540625,0.916667,0.624635,0.6325
qiskit-specific_experiment_tracking_using_mlflow_and_qprov,0.500000,0.000000,0.250000,0.438450,0.2050
qprov_provenance_taxonomy,0.855769,0.404167,0.833333,0.612704,0.5225


Average by (question_class, subdomain)


faithfulness  context_precision  context_recall  answer_relevance  factual_correctness
question_class subdomain                                                                                                                                 
fact_single    experiment_tracking_fundamentals                        0.000000           0.000000        0.000000          1.000000                 0.40
               mlflow_tracking_api                                     0.750000           0.000000        0.000000          0.000000                 0.29
               nisq_constraints_and_qsd_constraints                    1.000000           0.679167        1.000000          0.772702                 0.57
               qiskit-specific_experiment_tracking_using_mlflo...      0.750000           0.000000        0.000000          0.000000                 0.00
               qprov_provenance_taxonomy                               0.500000           0.333333        1.000000          0.604611                 0.00
reasoning      experiment_tracking_fundamentals                        0.000000           0.000000        1.000000          0.666977                 0.40
               mlflow_tracking_api                                     0.625000           0.805556        1.000000          0.719557                 0.80
               nisq_constraints_and_qsd_constraints                    1.000000           0.679167        0.666667          0.753315                 0.62
               qiskit-specific_experiment_tracking_using_mlflo...      0.000000           0.000000        0.000000          0.906727                 0.57
               qprov_provenance_taxonomy                               1.000000           0.950000        1.000000          0.974084                 0.67
summary        experiment_tracking_fundamentals                        0.833333           1.000000        0.750000          0.827169                 0.00
               mlflow_tracking_api                                     1.000000           1.000000        0.666667          0.630431                 0.33
               nisq_constraints_and_qsd_constraints                    1.000000           0.804167        1.000000          0.972523                 0.67
               qiskit-specific_experiment_tracking_using_mlflo...      1.000000           0.000000        1.000000          0.847071                 0.25
               qprov_provenance_taxonomy                               0.923077           0.333333        0.333333          0.872122                 0.42
unanswerable   experiment_tracking_fundamentals                        0.923077           0.000000        0.000000          0.530107                 0.00
               mlflow_tracking_api                                     0.500000           0.000000        1.000000          0.000000                 0.33
               nisq_constraints_and_qsd_constraints                    0.800000           0.000000        1.000000          0.000000                 0.67
               qiskit-specific_experiment_tracking_using_mlflo...      0.250000           0.000000        0.000000          0.000000                 0.00
               qprov_provenance_taxonomy                               1.000000           0.000000        1.000000          0.000000                 1.00